# 02 · Extraction de caractéristiques

SVC, KNN, arbres, Random Forest et XGBoost ne « voient » pas une image : ils travaillent sur un **tableau de nombres**.
On transforme donc chaque photo en un **vecteur de 554 caractéristiques** :

| Groupe | Taille | Ce que ça mesure |
|---|---|---|
| HOG | 432 | contours et orientations → la **forme** |
| Couleur | 27 | histogramme de teinte, noir/blanc/gris, moyennes HSV et Lab, **sur le vêtement uniquement** |
| Forme | 83 | silhouette en grille 8×10, rapport largeur/hauteur |
| Texture | 12 | motifs locaux (LBP) et force des gradients |

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, "..")                    # pour importer les modules de ml/
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

DATA = Path("../dataset")                   # dossier contenant styles.csv et images/

import feature_extraction as fe
from preprocess import build_dataframe, load_features
from PIL import Image
from skimage.color import rgb2gray
from skimage.feature import hog

df, images = build_dataframe(DATA, min_color_count=200)
sample = df.sample(1, random_state=3).iloc[0]
path = images / f"{sample.id}.jpg"
print(sample.articleType, sample.baseColour, sample.usage)

## Les étapes sur une image

In [ ]:
rgb = fe.load_rgb(path)
mask = fe.foreground_mask(rgb)
canon, cmask, aspect, fill, share = fe._crop_and_resize(rgb, mask)
gray = rgb2gray(canon / 255.0)
_, hog_img = hog(gray, orientations=9, pixels_per_cell=fe.HOG_CELL, cells_per_block=fe.HOG_BLOCK, visualize=True)

fig, axes = plt.subplots(1, 5, figsize=(16, 4))
for ax, img, title in zip(axes, [rgb, mask, canon, cmask, hog_img],
        ["1. image", "2. masque du vêtement", "3. recadrée 64×80", "4. masque recadré", "5. HOG (contours)"]):
    ax.imshow(img, cmap="gray" if img.ndim == 2 else None); ax.set_title(title); ax.axis("off")
plt.show()

In [ ]:
vec = fe.extract_features(path)
print("Taille du vecteur :", vec.shape)
for name, sl in fe.FEATURE_GROUPS.items():
    print(f"{name:8s} {sl.stop - sl.start:4d} valeurs")
hue = vec[fe.FEATURE_GROUPS["color"]][:fe.HUE_BINS]
plt.bar(range(fe.HUE_BINS), hue, color=plt.cm.hsv(np.linspace(0, 1, fe.HUE_BINS, endpoint=False)))
plt.title("Histogramme de teinte du vêtement"); plt.xlabel("teinte (12 classes)"); plt.show()

## Extraction sur tout le dataset (mise en cache)
La première fois cela prend quelques minutes ; ensuite le résultat est relu depuis `dataset/processed/`.

In [ ]:
import time
t0 = time.time()
X = load_features(df, images, cache_dir=DATA / "processed")
print("Matrice de caractéristiques X :", X.shape, f"({time.time() - t0:.0f} s)")

## Les classes sont-elles séparables ? (projection PCA en 2 dimensions)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
rows = np.random.RandomState(0).choice(len(df), min(3000, len(df)), replace=False)
coords = PCA(n_components=2, random_state=0).fit_transform(StandardScaler().fit_transform(X[rows]))
fig, ax = plt.subplots(figsize=(8, 6))
for t in sorted(df.articleType.unique()):
    m = (df.articleType.to_numpy()[rows] == t)
    ax.scatter(coords[m, 0], coords[m, 1], s=6, label=t, alpha=.6)
ax.legend(markerscale=3, fontsize=7, ncol=2); ax.set_title("Types de vêtements dans l'espace des caractéristiques (PCA)")
plt.show()

Des nuages qui se recouvrent partiellement annoncent des confusions entre classes proches : c'est ce que le notebook 03 mesure précisément.